## 1. Load & Inspect

Load the raw CSV and take a first look at its shape, dtypes, and missing values.

In [43]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

df = pd.read_csv('raw_loan_dataset.csv')

print("Shape:", df.shape)
df.head()

Shape: (103, 7)


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810,537.0,1.1,25800,Yes,No,No
1,96482,524.0,15.0,11200,Y,No,Yes
2,28478,NaN,5.4,12100,No,No,Yes
3,"$25,851",561.0,17.6,7000,Yes,No,Yes
4,38396,527.0,9.8,10700,No,No,Approved


In [44]:
print("=====informatoin======")
df.info()

=====informatoin======
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Approved          103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB


In [45]:
print("Missing values per column:")
df.isnull().sum()


Missing values per column:


Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64

## 2. Clean Currency Formatting


In [46]:
df["Income"]=df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"]=df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)


## 3. Fix Category Errors Before Imputation


In [47]:
yes_no_map = {
    "yes": "Yes",
    "y": "Yes",
    "yse": "Yes",
    "1": "Yes",
    "approved": "Yes",
    "no": "No",
    "n": "No",
    "0": "No",
    "rejected": "No",
}

df["HasCollateral"] = (
    df["HasCollateral"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(yes_no_map)
    .replace({"nan": np.nan})
)

df["PreviousDefaults"] = (
    df["PreviousDefaults"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(yes_no_map)
    .replace({"nan": np.nan})
)

df["Approved"] = (
    df["Approved"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(yes_no_map)
    .replace({"nan": np.nan})
)

print("Data after cleaning and normalizing:")
print(df.head())

Data after cleaning and normalizing:
     Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  108810.0        537.0              1.1     25800.0           Yes   
1   96482.0        524.0             15.0     11200.0           Yes   
2   28478.0          NaN              5.4     12100.0            No   
3   25851.0        561.0             17.6      7000.0           Yes   
4   38396.0        527.0              9.8     10700.0            No   

  PreviousDefaults Approved  
0               No       No  
1               No      Yes  
2               No      Yes  
3               No      Yes  
4               No      Yes  


In [48]:
print(df["HasCollateral"].value_counts(dropna=False))
print(df["PreviousDefaults"].value_counts(dropna=False))
print(df["Approved"].value_counts(dropna=False))

HasCollateral
No     51
Yes    50
NaN     2
Name: count, dtype: int64
PreviousDefaults
No     86
Yes    15
NaN     2
Name: count, dtype: int64
Approved
Yes    68
No     35
Name: count, dtype: int64


In [49]:
print(df["HasCollateral"].value_counts(dropna=False))
print(df["PreviousDefaults"].value_counts(dropna=False))
print(df["Approved"].value_counts(dropna=False))

HasCollateral
No     51
Yes    50
NaN     2
Name: count, dtype: int64
PreviousDefaults
No     86
Yes    15
NaN     2
Name: count, dtype: int64
Approved
Yes    68
No     35
Name: count, dtype: int64


## 4 Impute Missing Values


In [50]:
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])
print ("====messing vlue====")
print(df.isnull().sum())



====messing vlue====
Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64


## 5 Remove Duplicates



In [51]:
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]

print(f"Rows before: {before}")
print(f"Rows after:  {after}")
print(f"Duplicates removed: {before - after}")

Rows before: 103
Rows after:  100
Duplicates removed: 3


## 6. Outliers (IQR Capping)


In [52]:
num_cols = [
    "Income",
    "CreditScore",
    "LoanAmount",
    "EmploymentYears"
]
def cap_iqr(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return series.clip(lower=lower, upper=upper)

for col in num_cols:
    df[col] = cap_iqr(df[col])

df[num_cols].describe()

,Income,CreditScore,LoanAmount,EmploymentYears
count,100.00000,100.000000,100.000000,100.000000
mean,72220.22625,651.100000,25507.125000,12.654500
std,29179.39312,96.218239,14793.836616,7.011164
min,25851.00000,484.000000,5000.000000,0.500000
25%,47964.75000,576.000000,14400.000000,6.275000
50%,69460.50000,638.000000,20700.000000,12.550000
75%,95826.50000,730.500000,35125.000000,17.975000
max,167619.12500,920.000000,66212.500000,25.000000


## 7. Label Encoding

In [53]:
df['Approved'] = df['Approved'].map({'Yes': 1, 'No': 0})
df['HasCollateral'] = df['HasCollateral'].map({'Yes': 1, 'No': 0})
df['PreviousDefaults'] = df['PreviousDefaults'].map({'Yes': 1, 'No': 0})

df['Approved'].value_counts()


Approved
1    66
0    34
Name: count, dtype: int64

## 8. Class Balance Check

In [54]:
print("Counts:")
print(df['Approved'].value_counts())
print()
print("Proportions:")
print(df['Approved'].value_counts(normalize=True))


Counts:
Approved
1    66
0    34
Name: count, dtype: int64

Proportions:
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


## 9. Feature Engineering (No Leakage)
DebtToIncome = LoanAmount / Income   
IncomePerYearEmployed = Income / EmploymentYears


In [55]:
df['DebtToIncome'] = df['LoanAmount'] / df['Income']

# guard against division by zero for EmploymentYears == 0
df['IncomePerYearEmployed'] = df['Income'] / df['EmploymentYears'].replace(0, np.nan)
df['IncomePerYearEmployed'] = df['IncomePerYearEmployed'].fillna(df['IncomePerYearEmployed'].median())

df[['Income', 'LoanAmount', 'EmploymentYears', 'DebtToIncome', 'IncomePerYearEmployed']].head()

,Income,LoanAmount,EmploymentYears,DebtToIncome,IncomePerYearEmployed
0,108810.0,25800.0,1.1,0.237111,98918.181818
1,96482.0,11200.0,15.0,0.116084,6432.133333
2,28478.0,12100.0,5.4,0.424889,5273.703704
3,25851.0,7000.0,17.6,0.270783,1468.806818
4,38396.0,10700.0,9.8,0.278675,3917.959184


## 10. Feature Scaling

In [56]:
# Feature Scaling using RobustScaler


# Columns that should NOT be scaled
binary_cols = ["Approved", "HasCollateral", "PreviousDefaults"]

# Get all numeric columns
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Select numeric columns to scale
scale_cols = [col for col in numeric_cols if col not in binary_cols]

# Create the scaler
scaler = RobustScaler()

df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("RobustScaler applied successfully!")
print("Scaled columns:", scale_cols)

# Display the first five rows
df.head()

RobustScaler applied successfully!
Scaled columns: ['Income', 'CreditScore', 'EmploymentYears', 'LoanAmount', 'DebtToIncome', 'IncomePerYearEmployed']


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved,DebtToIncome,IncomePerYearEmployed
0,0.822149,-0.653722,-0.978632,0.246080,1,0,0,-0.445244,14.662142
1,0.564574,-0.737864,0.209402,-0.458384,1,0,1,-0.912749,0.129843
2,-0.856268,0.000000,-0.611111,-0.414958,0,0,1,0.280113,-0.052181
3,-0.911156,-0.498382,0.431624,-0.661037,1,0,1,-0.315175,-0.650043
4,-0.649046,-0.718447,-0.235043,-0.482509,0,0,1,-0.284688,-0.265208


## 11. Final Checks & Save

In [57]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 100 non-null    float64
 1   CreditScore            100 non-null    float64
 2   EmploymentYears        100 non-null    float64
 3   LoanAmount             100 non-null    float64
 4   HasCollateral          100 non-null    int64  
 5   PreviousDefaults       100 non-null    int64  
 6   Approved               100 non-null    int64  
 7   DebtToIncome           100 non-null    float64
 8   IncomePerYearEmployed  100 non-null    float64
dtypes: float64(6), int64(3)
memory usage: 7.8 KB


In [58]:
print(df.isnull().sum())

Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
IncomePerYearEmployed    0
dtype: int64


In [60]:
print("Missing values remaining:")
print(df.isnull().sum())
df.head()

Missing values remaining:
Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
IncomePerYearEmployed    0
dtype: int64


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved,DebtToIncome,IncomePerYearEmployed
0,0.822149,-0.653722,-0.978632,0.246080,1,0,0,-0.445244,14.662142
1,0.564574,-0.737864,0.209402,-0.458384,1,0,1,-0.912749,0.129843
2,-0.856268,0.000000,-0.611111,-0.414958,0,0,1,0.280113,-0.052181
3,-0.911156,-0.498382,0.431624,-0.661037,1,0,1,-0.315175,-0.650043
4,-0.649046,-0.718447,-0.235043,-0.482509,0,0,1,-0.284688,-0.265208


In [61]:
df.to_csv('clean_loan_dataset.csv', index=False)
print("Saved clean_loan_dataset.csv")

Saved clean_loan_dataset.csv
